In [ ]:
import os, glob, re, csv
import numpy as np
import pandas as pd
from scipy import stats, sparse
from scipy.stats import ttest_rel, ttest_ind
from scipy.sparse.csgraph import connected_components
from statsmodels.stats.multitest import fdrcorrection

from nilearn import plotting, image

In [ ]:
HPC        = '/nfs/roberts/pi/pi_il77/Nachshon/'
ATLAS      = 'difumo'
DATA_DIR   = f'{HPC}PSUB/preProc/corMatrix_byTrial_4_5/'
ATLAS_IMG  = f'{HPC}ROI/Atlas/difumo_2mm_maps_in_funcgrid.nii.gz' 
ATLAS_DIC  = f'{HPC}ROI/Atlas/labels_256_dictionary.csv'
PTSD_ROI   = 'ROI/difumo_ptsd_atlas_components_small.csv'
OUT_DIR    = "./ROI"
CONDITIONS = ['FULL', 'HOT60']
GROUPS     = ['PTSD', 'HC']
META_CSV   = '../../BehavioralData/clinical.csv'  
EPS        = 1e-7

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
headers    = ['roi_i', 'roi_j', 'roi_i_name', 'roi_j_name',
              'Yeo_7_i', 'Yeo_7_j', 'Yeo_17_i', 'Yeo_17_j', 'mean_r_A', 'mean_r_B', 
              'delta_r_AminusB', 'p_uncorrected', 'q_fdr', 'cohens_d'
             ]

In [ ]:
# Define your file path
coords_file = os.path.join(OUT_DIR, f"{ATLAS}_node_coords.csv")

if os.path.exists(coords_file):
    print(f"Loading coordinates from {coords_file}...")
    # Load the saved coordinates
    node_coords = np.loadtxt(coords_file, delimiter=',')
else:
    print(f"Calculating coordinates for {ATLAS} (this may take a while)...")
    labels_img = image.load_img(ATLAS_IMG)
    
    if ATLAS.lower() == 'difumo':
        # 4D probabilistic logic
        node_coords = plotting.find_probabilistic_atlas_cut_coords(labels_img)
    else:
        # 3D deterministic logic
        node_coords = plotting.find_parcellation_cut_coords(labels_img)
    
    # Save to OUT_DIR for next time
    np.savetxt(coords_file, node_coords, delimiter=',')
    print(f"Coordinates saved to {coords_file}")

p = len(node_coords)
print(f"ROIs (from atlas): {p}")

In [ ]:
# --- coords: choose API by atlas type ---
labels_img = image.load_img(ATLAS_IMG)
node_coords = plotting.find_probabilistic_atlas_cut_coords(labels_img)
p = len(node_coords)
print(f"ROIs (from atlas): {p}")

In [ ]:
# %% Collect subjects with all three condition files
all_files = sorted(glob.glob(os.path.join(DATA_DIR, f"*_{ATLAS}.csv")))

by_sub = {}
for f in all_files:
    base = os.path.basename(f).replace(f"_{ATLAS}.csv","")
    sub, _, cond = base.split("_", 2)
    sub = sub.split('-')[1]
    if cond in CONDITIONS:
        by_sub.setdefault(sub, {})[cond] = f


subs = sorted([s for s,d in by_sub.items() if all(c in d for c in CONDITIONS)])
print(f"Subjects with both conditions: n={len(subs)}")
assert len(subs) >= 2, "Need at least 2 subjects."

In [ ]:
meta = pd.read_csv(META_CSV)
meta["sub_id"] = meta["sub_id"].astype(str).str.strip()
meta["group"]  = meta["group"].str.upper().str.strip()

# subjects present in matrices
subs_all = sorted([s for s, d in by_sub.items() if any(c in d for c in CONDITIONS)])


# ensure normalized meta column is string
meta_ids = set(meta["sub_id"].astype(str))

missing_in_meta = [s for s in subs_all if s not in meta_ids]
if missing_in_meta:
    print("Subjects present in matrices but missing behavior:", missing_in_meta)


In [ ]:
# subjects in meta but missing any matrices (per condition we’ll check below)
present_in_meta = meta[meta["sub_id"].isin(by_sub.keys())].copy()
groups = present_in_meta.set_index("sub_id")["group"].to_dict()  # {sub: PTSD/HC}

In [ ]:
PTSD_atlas = pd.read_csv(PTSD_ROI)

roi_indices = (PTSD_atlas['Component'] - 1).tolist()
n_roi = len(roi_indices)

In [ ]:
sliced_data = {}

for sub in subs:
    sliced_data[sub] = {}
    for cond in CONDITIONS:
        file_path = by_sub[sub][cond]
        
        # Load the 256x256 matrix (assuming it's a CSV or similar)
        # If your files have headers/index, adjust pd.read_csv accordingly
        full_matrix = pd.read_csv(file_path, header=None).values 
        
        # Validate dimensions
        if full_matrix.shape != (256, 256):
            # If it's 256x257 or similar, you might need to drop an index column
            full_matrix = full_matrix[:256, :256]

        # 3. Slice the matrix using np.ix_ to get the 33x33 subset
        # This keeps only the rows and columns specified in roi_indices
        sub_matrix = full_matrix[np.ix_(roi_indices, roi_indices)]
        
        sliced_data[sub][cond] = sub_matrix

print(f"Successfully processed {len(subs)} subjects.")
print(f"New matrix shape: {sliced_data[subs[0]][CONDITIONS[0]].shape}")

In [ ]:
n_rois = sliced_data[subs[0]][CONDITIONS[0]].shape[0]
triu_idx = np.triu_indices(n_rois, k=1)
n_edges = len(triu_idx[0])
edge_data = []

for sub in subs:
    group_label = groups.get(sub)
    if group_label is None: continue # Skip if no group info
    
    for cond in CONDITIONS:
        matrix = sliced_data[sub][cond]
        # Extract the values of the 528 edges
        edges = matrix[triu_idx]
        
        for edge_val, edge_id in zip(edges, range(len(edges))):
            edge_data.append({
                'Subject': sub,
                'Group': group_label,
                'Condition': cond,
                'Edge_ID': edge_id,
                'Connectivity': edge_val
            })

df_long = pd.DataFrame(edge_data)

In [ ]:
# 1. Initialize storage for all p-values
p_vals = {'group': [], 'condition': [], 'interaction': []}
d_vals = {'group': [], 'condition': []}

print(f"Testing {n_edges} edges for all effects...")

for e in range(n_edges):
    edge_subset = df_long[df_long['Edge_ID'] == e]
    
    # --- A. Means and SS Calculations ---
    overall_mean = edge_subset['Connectivity'].mean()
    means = edge_subset.groupby(['Group', 'Condition'])['Connectivity'].mean()
    g_means = edge_subset.groupby('Group')['Connectivity'].mean()
    c_means = edge_subset.groupby('Condition')['Connectivity'].mean()
    
    # SS Error (Within-cell variance)
    ss_error = 0
    for g in GROUPS:
        for c in CONDITIONS:
            cell_data = edge_subset[(edge_subset['Group'] == g) & (edge_subset['Condition'] == c)]['Connectivity']
            ss_error += ((cell_data - means.loc[g, c])**2).sum()

    # SS Group, SS Condition, SS Interaction
    ss_group = sum([len(edge_subset[edge_subset['Group']==g]) * (g_means[g] - overall_mean)**2 for g in GROUPS])
    ss_cond = sum([len(edge_subset[edge_subset['Condition']==c]) * (c_means[c] - overall_mean)**2 for c in CONDITIONS])
    
    ss_inter = 0
    for g in GROUPS:
        for c in CONDITIONS:
            n_cell = len(edge_subset[(edge_subset['Group']==g) & (edge_subset['Condition']==c)])
            eff = (means.loc[g,c] - g_means[g] - c_means[c] + overall_mean)
            ss_inter += n_cell * (eff**2)

    # --- B. F-Tests ---
    df_g, df_c = len(GROUPS)-1, len(CONDITIONS)-1
    df_int = df_g * df_c
    df_err = len(edge_subset) - (len(GROUPS) * len(CONDITIONS))
    
    ms_err = ss_error / df_err
    p_vals['group'].append(stats.f.sf((ss_group/df_g)/ms_err, df_g, df_err))
    p_vals['condition'].append(stats.f.sf((ss_cond/df_c)/ms_err, df_c, df_err))
    p_vals['interaction'].append(stats.f.sf((ss_inter/df_int)/ms_err, df_int, df_err))

    # --- C. Cohen's d (Effect Size) ---
    def calc_d(df, factor):
        levs = df[factor].unique()
        v1, v2 = df[df[factor]==levs[0]]['Connectivity'], df[df[factor]==levs[1]]['Connectivity']
        sd_pool = np.sqrt(((len(v1)-1)*v1.var() + (len(v2)-1)*v2.var()) / (len(v1)+len(v2)-2))
        return abs((v1.mean() - v2.mean()) / sd_pool) if sd_pool != 0 else 0

    d_vals['group'].append(calc_d(edge_subset, 'Group'))
    d_vals['condition'].append(calc_d(edge_subset, 'Condition'))

In [ ]:
for effect in ['group', 'condition']:
    # Get raw p-values and calculate FDR (since this part still works)
    raw_ps = p_vals[effect]
    rejected, q_fdrs = fdrcorrection(raw_ps, alpha=0.05)
    
    # Check if we have anything to save
    if not any(rejected):
        print(f"No significant edges for {effect}. Skipping CSV.")
        continue
        
    fname = os.path.join(OUT_DIR, f"significant_{effect}_edges.csv")
    
    with open(fname, mode='w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        
        for e in range(n_edges):
            if rejected[e]:
                # Pull metadata from atlas
                row_i = PTSD_atlas.iloc[triu_idx[0][e]]
                row_j = PTSD_atlas.iloc[triu_idx[1][e]]
                
                # Calculate means for A vs B
                edge_subset = df_long[df_long['Edge_ID'] == e]
                # Dynamic labeling based on effect type
                factor = 'Group' if effect == 'group' else 'Condition'
                levs = list(GROUPS) if effect == 'group' else list(CONDITIONS)
                
                m_a = edge_subset[edge_subset[factor] == levs[0]]['Connectivity'].mean()
                m_b = edge_subset[edge_subset[factor] == levs[1]]['Connectivity'].mean()

                # Build the row
                writer.writerow({
                    'roi_i': triu_idx[0][e],
                    'roi_j': triu_idx[1][e],
                    'roi_i_name': row_i['Difumo_names'],
                    'roi_j_name': row_j['Difumo_names'],
                    'Yeo_7_i': row_i['Yeo_networks7'] if 'Yeo_networks7' in row_i.index else 'N/A',
                    'Yeo_7_j': row_j['Yeo_networks7'] if 'Yeo_networks7' in row_j.index else 'N/A',
                    'Yeo_17_i': row_i['Yeo_networks17'] if 'Yeo_networks17' in row_i.index else 'N/A',
                    'Yeo_17_j': row_j['Yeo_networks17'] if 'Yeo_networks17' in row_j.index else 'N/A',
                    'mean_r_A': m_a,
                    'mean_r_B': m_b,
                    'delta_r_AminusB': m_a - m_b,
                    'p_uncorrected': raw_ps[e],
                    'q_fdr': q_fdrs[e],
                    'cohens_d': d_vals[effect][e]
                })
                
    print(f"Successfully saved {sum(rejected)} edges to {fname}")

In [ ]:
summary_list = []
for effect in ['group', 'condition', 'interaction']:
    rej, q_fdr = fdrcorrection(p_vals[effect], alpha=0.05)
    
    # Calculate average d for main effects (interaction d is usually excluded or specialized)
    avg_d = np.mean(d_vals[effect]) if effect in d_vals else 0
    
    summary_list.append({
        'Measure': effect.capitalize(),
        'Number of Edges': rej.sum(),
        'Min q_fdr': f"{q_fdr.min():.4e}",
        'Avg Cohen\'s d': f"{avg_d:.3f}" if effect != 'interaction' else "N/A"
    })

df_summary = pd.DataFrame(summary_list)
df_summary

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p xarray